### Este es un web scrapging básico usando BeautifulSoup y requests para extraer informacion de productos en la web de Mercado libre.

- Se requiere realizar la busqueda de un link estatico ya con la consulta.
- Realizar la busqueda al sitio: https://listado.mercadolibre.com.co/teclado-gamer#D[A:Teclado%20gamer]

In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import re
import uuid
import base64

from dataclasses import dataclass, field, fields

def short_uuid() -> str:
    return base64.urlsafe_b64encode(uuid.uuid4().bytes).rstrip(b"=").decode("ascii")

@dataclass
class Item:
    name: str
    img_src: str
    last_price: str
    price: str
    instalment_id: str

    @classmethod
    def labels(cls):
        return [f.name for f in fields(cls)]

@dataclass
class Instalment:
    _id: str = field(default_factory=short_uuid,
                     init=False,
                     repr=False)
    number: str
    amount: str
    interest_rate: str

    _pattern = re.compile(
        r"^(\d+).+?(\$[\d\.]+).+?(\d+[.,]?\d*)%.+$",
        re.IGNORECASE
    )

    @property
    def id(self) -> str:
        return self._id

    @id.setter
    def id(self, value):
        raise AttributeError("El ID de un Instalment no puede modificarse.")

    @staticmethod
    def from_text(text: str) -> "Instalment":
        match = Instalment._pattern.search(text.strip())
        if not match:
            raise ValueError(f"Texto no válido: {text}")

        number = int(match.group(1))
        amount = match.group(2)
        interest_rate = match.group(3)

        return Instalment(
            number=number,
            amount=amount,
            interest_rate=interest_rate
        )
    
    @classmethod
    def labels(cls):
        return [f.name for f in fields(cls)]
    

def get_response(url, headers):
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return BeautifulSoup(response.content, 'html.parser')
    else:
        print(f"Error: Unable to fetch the webpage. Status code: {response.status_code}")
        return None
    
def get_text_using(tag, selector, class_name=None, attrs=None, none_str="N/A"):
    if class_name:
        found_tag = tag.find(selector, class_=class_name)
    elif attrs:
        found_tag = tag.find(selector, attrs=attrs)
    else:
        found_tag = tag.find(selector)
    return found_tag.text if found_tag else none_str    
    
def tag2item(tag) -> Item:
    title = tag.find('a', class_ = 'poly-component__title')

    price_box = tag.find('div', class_ = 'poly-component__price')
    last_price = get_text_using(price_box, 's')
    
    current_price_box = price_box.find('div', class_='poly-price__current')
    current_price = get_text_using(current_price_box, 'span', attrs={"role": "img"})
    str_installments = get_text_using(price_box, 'span', class_name='poly-price__installments')

    instalment = Instalment.from_text(str_installments)
    

    return Item(title.text, title["href"], last_price, current_price, instalment.id), instalment    


URL = "https://listado.mercadolibre.com.co/teclado-gamer#D[A:Teclado%20gamer]"
header = {
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36'
}

soup = get_response(URL, header)

item_boxes = soup.find_all('li', class_='ui-search-layout__item')
print(f"Number of items found: {len(item_boxes)}")
items, instalments = zip(*[tag2item(tag) for tag in item_boxes])

df_items = pd.DataFrame(items)
df_instalments = pd.DataFrame(instalments)

df_items.columns = Item.labels()
df_instalments.columns = Instalment.labels()

print(df_items.head())

print(df_instalments.head())



Number of items found: 50
                                                name  \
0                Combo Gamer Teclado Y Mouse Usb @gs   
1  Combo De Teclado Y Mouse Inalambrico Bt + Obse...   
2  Teclado Gamer Xtrike Me Mecanico 7 Colores Pc ...   
3  Kit de teclado y mouse inalámbrico Logitech MK...   
4  Teclado gamer Redragon Harpe Pro K503A RGB QWE...   

                                             img_src last_price     price  \
0  https://click1.mercadolibre.com.co/mclics/clic...        N/A   $39.900   
1  https://click1.mercadolibre.com.co/mclics/clic...    $57.000   $51.300   
2  https://www.mercadolibre.com.co/teclado-gamer-...   $169.000  $101.600   
3  https://www.mercadolibre.com.co/kit-de-teclado...        N/A   $99.000   
4  https://www.mercadolibre.com.co/teclado-gamer-...   $159.276  $109.900   

            instalment_id  
0  z0r19xwgTSmnqfIk2DhL6A  
1  flvXcaUQSVKvtPT_wGP7nQ  
2  MFMbttNHQb-yWU49qpkLOg  
3  udu1g78yRJueI_9ZvxZW1g  
4  lOBbB2l5Q120r51_h5iGlA  
      